# PD Open Field — Edge Preference & Motor Deficit 파이프라인 (테스트)

이 노트북은 **요힘빈(yohimbine) 주사 open field 데이터셋**(Zenodo, ETHZ-INS/Bohacek lab)
1~2개 샘플로 분석 파이프라인이 정상 작동하는지 확인하기 위한 테스트 노트북입니다.

추후 PD 모델 데이터로 교체하여 동일 파이프라인을 그대로 사용할 수 있습니다.

- 데이터 출처: https://zenodo.org/records/8188683 (CC-BY 4.0)
- Edge zone 정의: ETHZ-INS `DLCAnalyzer::AddOFTZones()` 기본값
  (`scale_center=0.5`, `scale_periphery=0.8`) 를 그대로 사용
  → 벽에서 arena 한 변 길이의 약 10% 이내가 **edge(periphery)**

## 이 노트북이 계산하는 것
1. **Motor deficit 지표**: 총 이동거리, 평균/최대 속도, 부동(immobility) 시간 비율
2. **Edge preference 지표**: periphery(가장자리) 체류 시간 비율, center 체류 시간 비율
3. Trajectory + zone 시각화, 비디오 프레임 위 궤적 오버레이


## 0. 환경 설정

In [ ]:
!pip -q install opencv-python-headless pandas numpy matplotlib requests tqdm


In [ ]:
import os, requests, zipfile, io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2
from tqdm import tqdm

DATA_DIR = "/content/data"
VIDEO_DIR = "/content/videos"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(VIDEO_DIR, exist_ok=True)

# 이 노트북과 같은 repo에 있는 src/ 헬퍼 함수 불러오기
# Colab에서 repo를 clone한 경우 아래 경로가 자동으로 잡힙니다.
import sys
REPO_SRC = "/content/PD-OpenField-EdgePreference/src"
if os.path.exists(REPO_SRC):
    sys.path.append(REPO_SRC)
else:
    # repo를 아직 clone하지 않았다면 GitHub에서 직접 clone
    !git clone https://github.com/famoushun98-create/PD-OpenField-EdgePreference.git /content/PD-OpenField-EdgePreference
    sys.path.append(REPO_SRC)

from dlc_io import load_dlc_csv
from oft_zones import calibrate_arena, classify_zone, edge_preference_summary
from motor_metrics import compute_speed, motor_summary


## 1. Zenodo metadata 다운로드 & 샘플 선택

전체 32개 녹음 중 빠른 테스트를 위해 **1~2개만** 골라서 진행합니다.
(saline 그룹 1개 + 고용량 그룹 1개를 기본 예시로 선택)

In [ ]:
RECORD_ID = "8188683"
api_url = f"https://zenodo.org/api/records/{RECORD_ID}"

r = requests.get(api_url, timeout=30)
r.raise_for_status()
record = r.json()

files = {f["key"]: f["links"]["self"] for f in record["files"]}
print("이 Zenodo record에 포함된 파일들:")
for k in files:
    print(" -", k)


In [ ]:
# METADATA.csv 다운로드 (그룹 정보가 들어있는 작은 파일)
meta_key = [k for k in files if "METADATA" in k.upper() and k.lower().endswith(".csv")][0]
meta_path = os.path.join(DATA_DIR, meta_key)

resp = requests.get(files[meta_key])
with open(meta_path, "wb") as f:
    f.write(resp.content)

metadata = pd.read_csv(meta_path)
metadata.head()


In [ ]:
# 그룹 컬럼명은 데이터셋마다 다를 수 있으니 먼저 컬럼을 확인하세요.
print(metadata.columns.tolist())


In [ ]:
# === 여기서 분석할 샘플 2개를 직접 지정하세요 ===
# metadata.head() 와 위 컬럼 목록을 보고, 파일명을 식별할 수 있는 컬럼/값을 선택해
# SAMPLE_IDS 에 실제 파일 식별자(예: 동물 ID, 파일명 등)를 넣어주세요.
SAMPLE_IDS = metadata.iloc[:2, 0].tolist()  # 우선 첫 2행을 임시 샘플로 사용
print("선택된 샘플:", SAMPLE_IDS)


## 2. Pose 데이터(csv) & 비디오 다운로드

`data.zip`, `Videos.zip` 전체를 한 번 받은 뒤, 위에서 고른 샘플 2개에 해당하는
파일만 추려서 사용합니다. (전체 32개를 다 풀지 않아도 됩니다.)

In [ ]:
def download_and_partial_extract(zip_key, out_dir, keep_substrings):
    """zip_key 파일을 다운로드한 뒤, 파일명에 keep_substrings 중 하나라도
    포함된 항목만 out_dir에 압축 해제합니다."""
    local_zip = os.path.join(DATA_DIR, zip_key)
    if not os.path.exists(local_zip):
        print(f"Downloading {zip_key} (시간이 걸릴 수 있습니다)...")
        with requests.get(files[zip_key], stream=True) as resp:
            resp.raise_for_status()
            total = int(resp.headers.get("content-length", 0))
            with open(local_zip, "wb") as f, tqdm(total=total, unit="B", unit_scale=True) as bar:
                for chunk in resp.iter_content(chunk_size=1 << 20):
                    f.write(chunk)
                    bar.update(len(chunk))

    extracted = []
    with zipfile.ZipFile(local_zip) as z:
        for name in z.namelist():
            if any(s in name for s in keep_substrings):
                z.extract(name, out_dir)
                extracted.append(os.path.join(out_dir, name))
    return extracted

data_zip_key = [k for k in files if k.lower() == "data.zip"][0]
video_zip_key = [k for k in files if k.lower() == "videos.zip"][0]

pose_files = download_and_partial_extract(data_zip_key, DATA_DIR, SAMPLE_IDS)
video_files = download_and_partial_extract(video_zip_key, VIDEO_DIR, SAMPLE_IDS)

print("Pose csv:", pose_files)
print("Video files:", video_files)


## 3. Pose 데이터 로드 & arena calibration

`tl, tr, br, bl` (top-left/top-right/bottom-right/bottom-left) bodypart가
csv에 있다고 가정합니다(DLCAnalyzer 관례). 없다면 아래에서 사용 가능한
bodypart 목록을 출력하니, `CORNER_POINTS` 와 `CENTER_BODYPART` 를 맞게 수정하세요.

In [ ]:
CORNER_POINTS = ("tl", "tr", "br", "bl")
CENTER_BODYPART = "bodycentre"   # 동물의 중심점으로 쓸 bodypart (없으면 'centre', 'center', 'body' 등 확인)
FPS = 25                          # 실제 녹화 fps로 수정하세요
LIKELIHOOD_CUTOFF = 0.9
MOVEMENT_CUTOFF_CM_S = 2.0        # 부동/이동 판정 기준 (calibration 전이면 px/s 기준으로 임시 사용)

assert len(pose_files) > 0, "pose csv를 찾지 못했습니다. SAMPLE_IDS를 다시 확인하세요."

results = []

for pose_path in pose_files:
    sample_name = os.path.basename(pose_path)
    print("\n===", sample_name, "===")

    data, bodyparts = load_dlc_csv(pose_path, likelihood_cutoff=LIKELIHOOD_CUTOFF)
    print("감지된 bodyparts:", bodyparts)

    if CENTER_BODYPART not in data:
        print(f"[경고] '{CENTER_BODYPART}' 가 없습니다. 위 목록에서 적절한 bodypart로 CENTER_BODYPART를 수정하세요.")
        continue

    arena = calibrate_arena(data, corner_points=CORNER_POINTS)
    x = data[CENTER_BODYPART]["x"].to_numpy()
    y = data[CENTER_BODYPART]["y"].to_numpy()

    zone = classify_zone(x, y, arena, scale_center=0.5, scale_periphery=0.8)
    edge_summary = edge_preference_summary(zone, fps=FPS)

    speed, dist_per_frame = compute_speed(x, y, fps=FPS)  # px 단위 (arena 실측 cm을 알면 px_to_cm 지정)
    motor = motor_summary(speed, dist_per_frame, fps=FPS, movement_cutoff=MOVEMENT_CUTOFF_CM_S)

    results.append({"sample": sample_name, **edge_summary, **motor})

    # --- 궤적 + zone 시각화 ---
    fig, ax = plt.subplots(figsize=(5, 5))
    colors = {"center": "tab:blue", "middle": "lightgray", "periphery": "tab:red"}
    for z in ["middle", "center", "periphery"]:
        mask = zone == z
        ax.scatter(x[mask], y[mask], s=2, c=colors[z], label=z)
    ax.set_title(sample_name)
    ax.invert_yaxis()
    ax.legend(markerscale=5)
    plt.show()

summary_df = pd.DataFrame(results)
summary_df


## 4. 비디오 프레임 위 궤적 오버레이 (육안 확인용)

In [ ]:
def overlay_trajectory_on_video(video_path, x, y, zone, out_path, max_frames=None):
    cap = cv2.VideoCapture(video_path)
    ok, frame = cap.read()
    if not ok:
        print(f"[경고] 비디오를 열 수 없습니다: {video_path}")
        return None

    h, w = frame.shape[:2]
    overlay = frame.copy()

    color_map = {"center": (255, 0, 0), "middle": (180, 180, 180), "periphery": (0, 0, 255)}
    n = len(x) if max_frames is None else min(max_frames, len(x))
    for i in range(1, n):
        if np.isnan(x[i]) or np.isnan(y[i]):
            continue
        pt1 = (int(x[i - 1]), int(y[i - 1]))
        pt2 = (int(x[i]), int(y[i]))
        cv2.line(overlay, pt1, pt2, color_map.get(zone[i], (0, 255, 0)), 1)

    cv2.imwrite(out_path, overlay)
    cap.release()
    return out_path

if video_files and len(results) > 0:
    for video_path in video_files:
        out_path = os.path.join(VIDEO_DIR, "overlay_" + os.path.basename(video_path) + ".png")
        # 같은 샘플의 x, y, zone을 매칭해서 사용하세요 (현재는 마지막으로 계산된 변수 재사용 예시)
        saved = overlay_trajectory_on_video(video_path, x, y, zone, out_path)
        if saved:
            img = cv2.imread(saved)
            plt.figure(figsize=(6, 6))
            plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            plt.axis("off")
            plt.title(os.path.basename(video_path))
            plt.show()
else:
    print("비디오 파일이 없거나 분석 결과가 없어 오버레이를 건너뜁니다.")


## 5. 결과 요약 표

`pct_time_periphery` (edge preference) 와 `total_distance` / `pct_time_immobile`
(motor deficit) 를 그룹별로 비교하면 됩니다.
PD 모델 데이터로 교체할 때는 1~3번 셀의 `RECORD_ID`, `SAMPLE_IDS`,
`CORNER_POINTS`, `CENTER_BODYPART` 만 맞게 바꾸면 동일 파이프라인을 그대로 쓸 수 있습니다.

In [ ]:
summary_df
